### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="heart_disease_va_long_beach",
    dataset_year="1989",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C52P4X",
    download_description="""
Get the UCI data.

wget https://archive.ics.uci.edu/static/public/45/heart+disease.zip && unzip heart+disease.zip processed.va.data && rm heart+disease.zip && mkdir -p local-data-warehouse/heart_disease_va_long_beach && mv processed.va.data local-data-warehouse/heart_disease_va_long_beach/
""",
    # References
    academic_reference_bibtex="""@article{detrano1989international,
  title={International application of a new probability algorithm for the diagnosis of coronary artery disease},
  author={Detrano, Robert and Janosi, Andras and Steinbrunn, Walter and Pfisterer, Matthias and Schmid, Johann-Jakob and Sandhu, Sarbjit and Guppy, Kern H and Lee, Stella and Froelicher, Victor},
  journal={The American journal of cardiology},
  volume={64},
  number={5},
  pages={304--310},
  year={1989},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="detrano1989international",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the processed version and the subset of 14 attributes used in the study and clinical practice.

- We encode missing values as np.nan instead of "?".
- We make the target binary (0=no heart disease, 1=heart disease). This follows the original study in attempting to distinguish presence (values 1,2,3,4) from absence (value 0).
- The dataset has one naturally occurring duplicate, which we do not drop.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="heart_disease_diagnosis",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="heart_disease_diagnosis",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

columns = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "num"
]

df = pd.read_csv(dataset_mold.path / "processed.va.data", header=None, names=columns)
print("Loaded data shape:", df.shape)

df = df.replace("?", np.nan)
df[["ca", "trestbps", "thalach", "oldpeak", "chol"]] = df[["ca", "trestbps", "thalach", "oldpeak", "chol"]].astype(float)
# Make target
df["heart_disease_diagnosis"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])

as_cat_type = ["thal", "slope", "exang", "restecg", "fbs", "cp", "sex", "heart_disease_diagnosis"]
df[as_cat_type] = df[as_cat_type].astype("category")


df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()